# Data Integration — Combining Player Datasets (2017–2026)

Combines two Kaggle FBref exports:
- **Historical** (`Top5_League_Players_2017to2024_dataset.csv`): 2017-18 to 2024-25, rich feature set (progression, key passes, aerial duels, xG/xAG)
- **Recent** (`players_data-2025_2026.csv`): 2025-26 only, narrower feature set (no progression/KP/aerials/xG — those tables aren't in this export)

**Why combine at all if the recent file is missing features?** Your outcome window can reach into 2025-26 (e.g. a player observed in 2023-24 needs a check 2 seasons later). The recent file fills that gap for basic stats, but the advanced features will be genuinely absent for 2025-26 rows — this is documented explicitly below rather than silently left as ordinary missingness.

## 0. Project setup

This notebook uses project-relative paths. Place the two source CSV files in `data/raw/fbref/`. Generated datasets are saved to `data/processed/`.

In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "fbref"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

HISTORICAL_FILE = RAW_DATA_DIR / "Top5_League_Players_2017to2024_dataset.csv"
RECENT_FILE = RAW_DATA_DIR / "players_data-2025_2026.csv"

for file_path in [HISTORICAL_FILE, RECENT_FILE]:
    if not file_path.exists():
        raise FileNotFoundError(
            f"Missing input file: {file_path}\n"
            "Download the FBref source file and place it in data/raw/fbref/."
        )

## 1. Load both sources

The historical file is semicolon-delimited with **commas as decimal separators** (European CSV format) — `decimal=','` is required or every numeric column silently loads as text. Verified separately that the file has zero structurally malformed rows, so `on_bad_lines` is kept only as a safety net, not because it's expected to trigger.

In [2]:
import pandas as pd

df_hist = pd.read_csv(
    HISTORICAL_FILE,
    sep=";",
    decimal=",",
    on_bad_lines="warn"
)
df_recent = pd.read_csv(
    RECENT_FILE,
    on_bad_lines="warn"
)

df_hist.columns = df_hist.columns.str.strip()
df_recent.columns = df_recent.columns.str.strip()

print(f"Historical dataset shape: {df_hist.shape}")
print(f"Recent dataset shape:     {df_recent.shape}")

Historical dataset shape: (22929, 178)
Recent dataset shape:     (2839, 102)


## 2. Standardise season labels

Historical file uses compact codes (`1920` → `2019-2020`); recent file is single-season and gets tagged directly.

In [3]:
def season_code_to_label(code):
    """'1920' -> '2019-2020'"""
    code = str(int(code))
    start_yy, end_yy = code[:2], code[2:]
    return f"20{start_yy}-20{end_yy}"

df_hist['Season'] = df_hist['season'].apply(season_code_to_label)
df_recent['Season'] = '2025-2026'

print("Historical seasons:", sorted(df_hist['Season'].unique()))
print("Recent seasons:", df_recent['Season'].unique())

Historical seasons: ['2017-2018', '2018-2019', '2019-2020', '2020-2021', '2021-2022', '2022-2023', '2023-2024', '2024-2025']
Recent seasons: ['2025-2026']


## 3. Compare column alignment BEFORE combining

Important: at this point the two files still use different naming conventions for the *same* concepts (`player` vs `Player`, `team` vs `Squad`). If concatenated as-is, these would stay as separate columns and silently fragment player identity across the two sources — this is checked explicitly here, then fixed in the next cell before any row-combining happens.

In [4]:
cols_hist = set(df_hist.columns)
cols_recent = set(df_recent.columns)

common_cols = cols_hist.intersection(cols_recent)
only_in_hist = cols_hist - cols_recent
only_in_recent = cols_recent - cols_hist

print(f"Common Columns ({len(common_cols)}): {sorted(common_cols)}\n")
print(f"Columns only in Historical ({len(only_in_hist)}) — includes the advanced features:")
print(sorted(only_in_hist))
print(f"\nColumns only in Recent ({len(only_in_recent)}):")
print(sorted(only_in_recent))

Common Columns (1): ['Season']

Columns only in Historical (178) — includes the advanced features:
['1/3_', '90s_', 'Aerial Duels_Lost', 'Aerial Duels_Won', 'Aerial Duels_Won%', 'Ast_', 'Att_', 'Blocks_Blocks', 'Blocks_Pass', 'Blocks_Sh', 'Carries_1/3', 'Carries_CPA', 'Carries_Carries', 'Carries_Dis', 'Carries_Mis', 'Carries_PrgC', 'Carries_PrgDist', 'Carries_TotDist', 'Challenges_Att', 'Challenges_Lost', 'Challenges_Tkl', 'Challenges_Tkl%', 'Clr_', 'Corner Kicks_In', 'Corner Kicks_Out', 'Corner Kicks_Str', 'CrsPA_', 'Err_', 'Expected_A-xAG', 'Expected_G-xG', 'Expected_np:G-xG', 'Expected_npxG', 'Expected_npxG+xAG', 'Expected_npxG/Sh', 'Expected_xA', 'Expected_xAG', 'Expected_xG', 'GCA Types_Def', 'GCA Types_Fld', 'GCA Types_PassDead', 'GCA Types_PassLive', 'GCA Types_Sh', 'GCA Types_TO', 'GCA_GCA', 'GCA_GCA90', 'Int_', 'KP_', 'Long_Att', 'Long_Cmp', 'Long_Cmp%', 'Medium_Att', 'Medium_Cmp', 'Medium_Cmp%', 'Outcomes_Blocks', 'Outcomes_Cmp', 'Outcomes_Off', 'PPA_', 'Pass Types_CK', 'Pass

## 4. Align schemas onto shared column names

Only columns that exist conceptually in **both** sources get renamed onto a shared name. Columns unique to `df_hist` (progression, key passes, aerial duels, xG/xAG) are left as-is — they will be `NaN` for every 2025-26 row after concatenation, which is expected and checked explicitly in Section 6, not hidden.

In [5]:
hist_rename = {
    'player': 'Player', 'nation_': 'Nation', 'pos_': 'Pos', 'age_': 'Age',
    'team': 'Team', 'league': 'Comp',
    'Playing Time_MP': 'MP', 'Playing Time_Min': 'Min', 'Playing Time_90s': '90s',
    'Performance_Gls': 'Gls', 'Performance_Ast': 'Ast', 'Performance_G+A': 'G+A',
    'Tackles_TklW': 'TklW', 'Int_': 'Int',
}
df_hist = df_hist.rename(columns=hist_rename)

recent_rename = {
    'Squad': 'Team',
    # Player, Nation, Pos, Age, Comp, MP, Min, 90s, Gls, Ast, G+A, TklW, Int
    # already match df_hist's renamed columns directly
}
df_recent = df_recent.rename(columns=recent_rename)

# Re-check alignment after renaming
common_after = set(df_hist.columns).intersection(set(df_recent.columns))
print(f"Common columns after renaming: {len(common_after)}")
print(sorted(common_after))

Common columns after renaming: 15
['90s', 'Age', 'Ast', 'Comp', 'G+A', 'Gls', 'Int', 'MP', 'Min', 'Nation', 'Player', 'Pos', 'Season', 'Team', 'TklW']


## 5. Combine (row-wise)

In [6]:
df_combined = pd.concat([df_hist, df_recent], ignore_index=True, sort=False)

assert len(df_combined) == len(df_hist) + len(df_recent), "Row count mismatch after concatenation!"

print(f"Combined Dataset Shape: {df_combined.shape}")
print("\nSeasons present:", sorted(df_combined['Season'].unique()))

Combined Dataset Shape: (25768, 267)

Seasons present: ['2017-2018', '2018-2019', '2019-2020', '2020-2021', '2021-2022', '2022-2023', '2023-2024', '2024-2025', '2025-2026']


## 6. Confirm the expected 2025-26 feature gap

This should show ~100% missing for advanced features on 2025-26 rows specifically — confirming the gap is exactly as expected from the recent file's narrower schema, not a merge error.

In [7]:
advanced_cols = ['Progression_PrgC', 'Progression_PrgP', 'Progression_PrgR',
                  'KP_', 'Aerial Duels_Won', 'Aerial Duels_Lost',
                  'Expected_xG', 'Expected_xAG']

print("Missing % for advanced features, 2025-26 rows only:")
recent_mask = df_combined['Season'] == '2025-2026'
for col in advanced_cols:
    if col in df_combined.columns:
        pct_missing = df_combined.loc[recent_mask, col].isna().mean()
        print(f"  {col}: {pct_missing:.1%} missing")
    else:
        print(f"  {col}: not found in combined columns — check naming")

print("\nOverall missing-value summary (top 10 columns):")
missing_summary = pd.DataFrame({
    'Missing Values': df_combined.isnull().sum(),
    'Percentage': (df_combined.isnull().sum() / len(df_combined)) * 100
})
missing_summary.sort_values(by='Missing Values', ascending=False).head(10)

Missing % for advanced features, 2025-26 rows only:
  Progression_PrgC: 100.0% missing
  Progression_PrgP: 100.0% missing
  Progression_PrgR: 100.0% missing
  KP_: 100.0% missing
  Aerial Duels_Won: 100.0% missing
  Aerial Duels_Lost: 100.0% missing
  Expected_xG: 100.0% missing
  Expected_xAG: 100.0% missing

Overall missing-value summary (top 10 columns):


,Missing Values,Percentage
CS%,25580,99.270413
Save%,25578,99.262651
PKatt_stats_keeper,25574,99.247128
W,25574,99.247128
D,25574,99.247128
L,25574,99.247128
Comp_stats_keeper,25574,99.247128
Age_stats_keeper,25574,99.247128
GA,25574,99.247128
Saves,25574,99.247128


## 7. Observation-scope filter

Filters to your dissertation's actual scope (Big 5 leagues, 2019-20 to 2023-24, age 18-24). The **full** `df_combined` (including 2025-26) is kept separately for outcome-window lookups (e.g. checking a 2023-24 observation's status 2 seasons later) — only `observation_scope` is restricted to seasons players are actually *drawn from*.

In [8]:
scope_seasons = ['2019-2020', '2020-2021', '2021-2022', '2022-2023', '2023-2024']
big5 = ['Premier League', 'La Liga', 'Bundesliga', 'Serie A', 'Ligue 1']

observation_scope = df_combined[
    (df_combined['Season'].isin(scope_seasons)) &
    (df_combined['Comp'].str.contains('|'.join(big5), na=False)) &
    (df_combined['Age'].between(18, 24))
].copy()

print(f"Rows in observation scope (2019-20 to 2023-24, Big 5, age 18-24): {len(observation_scope)}")
print(f"Full combined dataset (all seasons, for outcome-window lookups): {len(df_combined)}")

Rows in observation scope (2019-20 to 2023-24, Big 5, age 18-24): 6231
Full combined dataset (all seasons, for outcome-window lookups): 25768


## 8. Save outputs

In [9]:
full_csv = PROCESSED_DATA_DIR / "Top5_League_Players_2017to2026_combined.csv"
full_parquet = PROCESSED_DATA_DIR / "Top5_League_Players_2017to2026_combined.parquet"
scope_csv = PROCESSED_DATA_DIR / "Top5_League_Players_observation_scope.csv"

df_combined.to_csv(full_csv, index=False)
# Parquet is optional; requires pyarrow or fastparquet.
try:
    df_combined.to_parquet(full_parquet, index=False)
    print(f"Saved: {full_parquet}")
except (ImportError, ModuleNotFoundError):
    print("Parquet export skipped: install pyarrow or fastparquet to enable it.")

observation_scope.to_csv(scope_csv, index=False)

print("Successfully exported:")
print(f" - {full_csv}")
print(f" - {scope_csv}")

Parquet export skipped: install pyarrow or fastparquet to enable it.


Successfully exported:
 - /mnt/data/football-player-development-prediction-repro/data/processed/Top5_League_Players_2017to2026_combined.csv
 - /mnt/data/football-player-development-prediction-repro/data/processed/Top5_League_Players_observation_scope.csv
